# Murshid (مُرشد) — A University Student Services Agent

**Author:** «Your full name — submissions without a name are not graded»
**Training programme:** SDAIA Academy — Building Agentic AI Systems
**Cohort dates:** «e.g. 12–16 October 2026»

## Declared capstone track: **A — Supervisor + Workers**

Track **C** (multi-source routing) and Track **B** (handoff / human escalation)
are also implemented, because rubric sections 3 and 5 require their mechanisms
regardless of the declared track.

---

### What this system does

A student asks one free-text question, in Arabic or English. Murshid:

1. **Routes** it with an LLM classifier to Academic Affairs, Campus Services,
   both, or "this is a request to *file* something".
2. **Answers** it from that specialist's own private vector store.
3. **Remembers** the student across separate conversations — language, major,
   question count — in a Store, not a message list.
4. **Pauses for a human** before anything irreversible. A course withdrawal
   cannot be undone after the Registrar processes it, so an advisor approves,
   edits, or rejects it first.

### How to run this notebook

**Restart the runtime and run every cell, in order, from the top.** Cells run
out of order are the single most common cause of a silently broken pipeline.

Section 7 is a **stop gate**: if the retrieval smoke test fails, fix it before
running anything below it.

---
# 1 · Install

In [ ]:
# Colab ships a chromadb that conflicts with its own preinstalled
# opentelemetry/numpy. Pin them BEFORE importing anything else.
# (Skip this cell if you are running locally with requirements.txt installed.)

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    !pip uninstall -y -q chromadb opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp opentelemetry-exporter-otlp-proto-grpc numpy
    !pip install -qU chromadb==0.4.18 opentelemetry-api==1.42.1 opentelemetry-sdk==1.42.1 "numpy<2.0.0"
    !pip install -qU langchain langchain-core langchain-community langchain-groq langchain-text-splitters langchain-huggingface sentence-transformers langgraph langgraph-supervisor langsmith pydantic

print("IN_COLAB =", IN_COLAB)

In [ ]:
import os
os.environ["CHROMA_SERVER_NO_TELEMETRY"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

---
# 2 · Secrets and data

**Never paste an API key into a cell.** In Colab use the Secrets panel (the key
icon in the left sidebar); locally use a `.env` file that is git-ignored. A key
committed to git stays in the history even after you delete the line — if that
happens, revoke and rotate it at the provider immediately.

In [ ]:
import os

def load_secret(name: str, required: bool = True) -> str | None:
    """Read a secret from Colab Secrets, then the environment, then .env."""
    val = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
        except Exception:
            val = None
    if not val:
        val = os.environ.get(name)
    if not val:
        try:
            from dotenv import load_dotenv
            load_dotenv()
            val = os.environ.get(name)
        except ImportError:
            pass
    if not val and required:
        raise RuntimeError(
            f"{name} is not set. In Colab add it to the Secrets panel "
            f"(key icon, left sidebar) and enable notebook access. "
            f"Locally, put it in a .env file."
        )
    return val


os.environ["GROQ_API_KEY"] = load_secret("GROQ_API_KEY")
print("GROQ_API_KEY loaded:", bool(os.environ.get("GROQ_API_KEY")))

In [ ]:
# --- get the 16 knowledge-base documents ------------------------------------
# Three ways this can work, tried in order. You do NOT need a GitHub account.
#
#   1. Running locally in the repo   -> data/ is already here, nothing to do
#   2. Colab, zip uploaded           -> upload murshid-capstone.zip via the
#                                       Files panel (folder icon, left sidebar),
#                                       or run this cell and pick the file
#   3. Colab, repo on GitHub         -> set REPO_URL below and it clones

REPO_URL = ""      # optional — only if you have pushed this to GitHub

from pathlib import Path
import zipfile

def find_data_dir():
    for c in [Path("data"), Path("../data"), Path("murshid-capstone/data"),
              *Path(".").glob("*/data")]:
        if (c / "academic").is_dir() and (c / "campus").is_dir():
            return c.resolve()
    return None


DATA_DIR = find_data_dir()

# --- 2. a zip sitting in the working directory -------------------------------
if DATA_DIR is None:
    for z in Path(".").glob("*.zip"):
        print(f"found {z.name} — extracting")
        zipfile.ZipFile(z).extractall(".")
    DATA_DIR = find_data_dir()

# --- 2b. Colab: offer an upload dialog ---------------------------------------
if DATA_DIR is None and IN_COLAB:
    print("No data/ found. Upload murshid-capstone.zip when prompted.")
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            zipfile.ZipFile(name).extractall(".")
    DATA_DIR = find_data_dir()

# --- 3. clone from GitHub, if you have set REPO_URL --------------------------
if DATA_DIR is None and REPO_URL.strip():
    !git clone -q {REPO_URL} murshid-capstone
    DATA_DIR = find_data_dir()

if DATA_DIR is None:
    raise RuntimeError(
        "Could not find the data/ folder.\n"
        "Upload murshid-capstone.zip to this runtime (Files panel, folder icon "
        "in the left sidebar) and re-run this cell."
    )

print("data directory:", DATA_DIR)
print("academic files:", len(list((DATA_DIR / "academic").glob("*.md"))))
print("campus   files:", len(list((DATA_DIR / "campus").glob("*.md"))))

---
# 3 · LangSmith tracing — rubric §8

The tracing flag is **`LANGCHAIN_TRACING_V2`**, set to the *string* `"true"`.

`LANGSMITH_TRACING_V2` is **not a real variable**. It produces no trace, no
error, and an empty project page. This is the single most common way to lose
this section.

This notebook runs fine without a LangSmith key — the cell below degrades
gracefully so you can build everything else first and add tracing later.

In [ ]:
LANGSMITH_KEY = load_secret("LANGSMITH_API_KEY", required=False)
TRACING_ON = False

if LANGSMITH_KEY:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"      # <- EXACT name, string "true"
    os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_KEY
    os.environ["LANGCHAIN_PROJECT"] = "murshid-capstone"

    # Verify the key BEFORE running the agent. This turns a silent 401 into a
    # loud one, instead of an empty project page you discover during the demo.
    from langsmith import Client
    try:
        client = Client()
        list(client.list_projects(limit=1))
        TRACING_ON = True
        print("LangSmith OK — key valid, project:", os.environ["LANGCHAIN_PROJECT"])
    except Exception as e:
        print("LangSmith key was REJECTED:", type(e).__name__, e)
        print("Get a new one at smith.langchain.com -> Settings -> API Keys.")
        os.environ["LANGCHAIN_TRACING_V2"] = "false"
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("No LANGSMITH_API_KEY found — tracing is OFF.")
    print("Everything else in this notebook still runs.")
    print("Add the key to Secrets and re-run this cell to enable section 8.")

print("LANGCHAIN_TRACING_V2 =", os.environ.get("LANGCHAIN_TRACING_V2"))

---
# 4 · Model and embeddings

**Model choice.** Groq decommissioned `llama-3.3-70b-versatile` — the
model used throughout the course lessons — on **16 August 2026**. This cell
therefore queries the account for the models it can actually use and picks
the first available from a preference list, instead of hardcoding an ID that
may be retired again. The printed list is a record of what was available on
the day this notebook was run.

**Embedding model choice.** The course lessons use
`sentence-transformers/all-mpnet-base-v2`, which is English-only. Murshid's
students ask in Arabic *and* English against a knowledge base written in English
with Arabic summaries, so this project uses
`paraphrase-multilingual-mpnet-base-v2` instead — same family, same cost
(free, runs locally, no API key), but it handles cross-lingual retrieval.

That swap is a deliberate design decision and belongs in the write-up.

In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from groq import Groq

# Groq decommissioned llama-3.3-70b-versatile on 16 August 2026 (the model the
# course lessons use). Rather than hardcode a replacement that may also be
# retired later, ask the account which models it can actually use and take the
# first supported one in order of preference.
available = {m.id for m in Groq().models.list().data}

PREFERRED = [
    "openai/gpt-oss-120b",      # Groq flagship production model, tool calling
    "qwen/qwen3.6-27b",         # Groq's other named replacement
    "openai/gpt-oss-20b",       # smaller, faster fallback
    "llama-3.3-70b-versatile",  # the course default, if your account still has it
]

MODEL = next((m for m in PREFERRED if m in available), None)
if MODEL is None:
    raise RuntimeError(
        "None of the preferred models are available to this key.\n"
        f"Your account can use: {sorted(available)}\n"
        "Pick a chat model from that list and set MODEL manually."
    )

print("chat models available to this key:")
for m in sorted(x for x in available if "whisper" not in x):
    print("   ", m, "  <-- using this" if m == MODEL else "")

llm = ChatGroq(model=MODEL, temperature=0)

# Downloads once (~1 GB), cached afterwards. No API key needed.
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

print("\nllm       :", MODEL)
print("embeddings:", embeddings.model_name)
print("\nsanity check:", llm.invoke("Reply with exactly: ready").content)

---
# 5 · RAG stage 1–2: Load → Split — rubric §3

Sixteen markdown documents describing a fictional institution, "Al-Noor
University", split into two disjoint corpora:

- `data/academic/` — grading, appeals, attendance, withdrawal, probation,
  transcripts, graduation, exam conflicts
- `data/campus/` — library, IT helpdesk, portal access, housing, dining,
  careers, wellbeing, clubs

The splitter separates on markdown headings first, so a regulation is not cut
mid-clause.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

def load_corpus(folder: Path):
    docs = DirectoryLoader(
        str(folder), glob="**/*.md",
        loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"},
    ).load()
    for d in docs:
        d.metadata["source"] = Path(d.metadata.get("source", "unknown")).name
        d.metadata["collection"] = folder.name
    return docs

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
    separators=["\n## ", "\n### ", "\n\n", "\n", " ", ""],
)

academic_docs = load_corpus(DATA_DIR / "academic")
campus_docs   = load_corpus(DATA_DIR / "campus")

academic_chunks = splitter.split_documents(academic_docs)
campus_chunks   = splitter.split_documents(campus_docs)

print(f"academic: loaded {len(academic_docs)} documents -> {len(academic_chunks)} chunks")
print(f"campus  : loaded {len(campus_docs)} documents -> {len(campus_chunks)} chunks")
print("\nexample chunk:")
print(" source:", academic_chunks[0].metadata["source"])
print(" text  :", academic_chunks[0].page_content[:200], "...")

---
# 6 · RAG stage 3–5: Embed → Store → Retrieve — rubric §3

**Two separate Chroma collections.** This is the part that makes routing
meaningful. Two retrievers pointed at the same store would make the routing
decision change nothing, and a grader spots that immediately.

In [ ]:
from langchain_community.vectorstores import Chroma

academic_store = Chroma.from_documents(
    academic_chunks, embeddings, collection_name="academic")
campus_store = Chroma.from_documents(
    campus_chunks, embeddings, collection_name="campus")

academic_retriever = academic_store.as_retriever(search_kwargs={"k": 3})
campus_retriever   = campus_store.as_retriever(search_kwargs={"k": 3})

print("academic collection:", academic_store._collection.count(), "vectors")
print("campus   collection:", campus_store._collection.count(), "vectors")

---
# 7 · STOP GATE — does retrieval actually work? — rubric §3

Ask questions whose answers are **verbatim** in the documents. If a retriever
returns nothing, or the expected fact is missing from the top-3, the pipeline is
broken no matter how correct the code looks — and nothing below this point is
worth running.

The second test proves the two stores are genuinely **isolated**: the academic
store must not know the library hours, and the campus store must not know the
appeals deadline.

In [ ]:
SMOKE_TESTS = [
    ("academic", academic_retriever, "grade appeal deadline",           "15"),
    ("academic", academic_retriever, "minimum attendance percentage",   "75"),
    ("academic", academic_retriever, "course withdrawal deadline",      "week 10"),
    ("academic", academic_retriever, "credit hours required to graduate", "132"),
    ("campus",   campus_retriever,   "library hours during finals week", "2:00 AM"),
    ("campus",   campus_retriever,   "where is the IT helpdesk",         "Building 4"),
    ("campus",   campus_retriever,   "student portal account lockout",   "30 minutes"),
    ("campus",   campus_retriever,   "how many members to start a club", "15"),
]

all_ok = True
print("--- retrieval smoke test ---")
for collection, retriever, query, expected in SMOKE_TESTS:
    docs = retriever.invoke(query)
    if not docs:
        print(f"  FAIL [{collection:8}] {query!r} -> retriever returned NOTHING")
        all_ok = False
        continue
    joined = " ".join(d.page_content for d in docs)
    hit = expected.lower() in joined.lower()
    all_ok = all_ok and hit
    print(f"  {'PASS' if hit else 'FAIL'} [{collection:8}] {query!r}")
    print(f"        expected {expected!r} | top hit: {docs[0].metadata['source']}")

print("\nSMOKE TEST:", "PASSED" if all_ok else "FAILED — fix this before continuing")

In [ ]:
print("--- cross-store isolation test ---\n")

a = academic_retriever.invoke("library opening hours during finals week")
print("academic store, asked about LIBRARY HOURS:")
print("  returned:", [d.metadata["source"] for d in a])
print("  -> correct behaviour is academic regulations, NOT library_hours.md\n")

c = campus_retriever.invoke("grade appeal deadline form AR-12")
print("campus store, asked about GRADE APPEALS:")
print("  returned:", [d.metadata["source"] for d in c])
print("  -> correct behaviour is service pages, NOT grade_appeals.md")

In [ ]:
# What a retrieved chunk actually looks like — proof the text is real.
for d in academic_retriever.invoke("How long do I have to appeal a grade?"):
    print(f"[{d.metadata['source']}]")
    print(d.page_content[:300])
    print("-" * 70)

---
# 8 · Tools — rubric §1

Every tool below does **real work on its arguments**. A function that ignores
its inputs and returns a fixed f-string is not a tool call, and it is the most
commonly penalised mistake in this section.

In [ ]:
from langchain_core.tools import tool

PROGRAMME_TOTAL_CREDITS = 132     # from data/academic/graduation_requirements.md


@tool
def compute_gpa(grades: list[float], credits: list[int]) -> str:
    """Compute a credit-weighted GPA from grade points and matching credit hours.

    Use when a student gives their grades and wants a GPA or their academic
    standing. grades are 4.00-scale points, credits are the credit hours of the
    matching course.
    """
    if len(grades) != len(credits):
        raise ValueError(
            f"grades ({len(grades)}) and credits ({len(credits)}) must match")
    if not credits or sum(credits) == 0:
        raise ValueError("credits must contain at least one non-zero value")

    total_points = sum(g * c for g, c in zip(grades, credits))
    total_credits = sum(credits)
    gpa = total_points / total_credits

    if gpa >= 3.75:
        standing = "Dean's List range"
    elif gpa >= 2.00:
        standing = "Good Standing"
    else:
        standing = "below the 2.00 threshold — academic probation applies"

    return (f"GPA {gpa:.2f} across {total_credits} credit hours "
            f"({total_points:.1f} total grade points) — {standing}.")


@tool
def credits_to_graduate(completed_credits: int) -> str:
    """How many credit hours a student still needs, and roughly how many terms.

    Use when a student asks how much is left before they graduate.
    """
    if completed_credits < 0:
        raise ValueError("completed_credits cannot be negative")
    remaining = max(PROGRAMME_TOTAL_CREDITS - completed_credits, 0)
    if remaining == 0:
        return (f"{completed_credits} of {PROGRAMME_TOTAL_CREDITS} credit hours "
                f"complete — the credit-hour requirement is already met.")
    terms = -(-remaining // 15)          # ceiling division at 15 cr/term
    return (f"{remaining} credit hours remaining of {PROGRAMME_TOTAL_CREDITS} "
            f"— about {terms} more full-time term(s) at 15 hours each.")


@tool
def search_academic_regulations(query: str) -> str:
    """Search Al-Noor University academic regulations.

    Covers grading and GPA, grade appeals, attendance, course withdrawal,
    academic probation, transcripts, graduation requirements, and examinations.
    """
    docs = academic_retriever.invoke(query)
    if not docs:
        return "No matching academic regulation found."
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)


@tool
def search_campus_services(query: str) -> str:
    """Search Al-Noor University campus services documentation.

    Covers the library, IT helpdesk, student portal access, housing, dining,
    the careers office, wellbeing and counselling, and clubs and societies.
    """
    docs = campus_retriever.invoke(query)
    if not docs:
        return "No matching campus service page found."
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)


TOOLS = [compute_gpa, credits_to_graduate,
         search_academic_regulations, search_campus_services]
for t in TOOLS:
    print(f"{t.name:32} {list(t.args.keys())}")

In [ ]:
# Proof the tools do real work: different arguments -> different results.
print(compute_gpa.invoke({"grades": [4.0, 3.5, 3.0], "credits": [3, 3, 4]}))
print(compute_gpa.invoke({"grades": [2.0, 1.5, 1.0], "credits": [3, 3, 4]}))
print(credits_to_graduate.invoke({"completed_credits": 87}))
print(credits_to_graduate.invoke({"completed_credits": 132}))

---
# 9 · The router — rubric §1 (structured output) and §2 (routing)

A router is an LLM call with a constrained return type. Nothing more.

`Literal` does two jobs at once: it stops the model emitting a destination the
workflow has no branch for, and it keeps the type and the branches from drifting
apart. That constraint is also the cheapest guardrail in the system.

Two fields beyond the destination earn their place:

- `language` — lets the answer come back in the language the student used.
- `confidence` — a **second, genuine** human-in-the-loop trigger. A low-confidence
  question is escalated to a person instead of guessed at.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class MurshidRoute(BaseModel):
    """Where a student question should be handled."""

    destination: Literal["academic", "campus", "both", "action"] = Field(
        description=(
            "academic  — grades, GPA, exams, attendance, transcripts, probation, "
            "withdrawal RULES and deadlines, graduation requirements; "
            "campus    — library, IT helpdesk, student portal login, housing, "
            "dining, careers, wellbeing, clubs; "
            "both      — the question genuinely needs each source; "
            "action    — the student is asking to actually FILE or SUBMIT "
            "something now: a withdrawal, a grade appeal, a formal request. "
            "Asking HOW to withdraw is 'academic'; asking TO withdraw is 'action'."
        )
    )
    reason: str = Field(description="One short sentence justifying the choice")
    language: Literal["ar", "en"] = Field(
        description="The language the student wrote in")
    confidence: Literal["high", "low"] = Field(
        description=("low when the question is ambiguous, underspecified, or "
                     "you are unsure which source holds the answer"))


router = llm.with_structured_output(MurshidRoute)

d = router.invoke("I was double-charged for my housing fee")
print(d)

### Why this is not keyword matching

The seven questions below are chosen because a keyword router gets at least
three of them wrong:

- The two Arabic questions contain **no English keyword at all**.
- *"I was told my attendance is short but I have a medical note"* contains
  neither "attendance policy" nor "appeal" as a literal phrase.
- *"How do I appeal a grade, and where is the IT helpdesk?"* matches **both**
  categories, which a first-match `if/elif` chain resolves arbitrarily.
- *"How do I withdraw from a course?"* and *"I want to withdraw from STAT301"*
  share every keyword but need **different destinations** — one is a question,
  one is a request to act.

In [ ]:
ROUTING_TESTS = [
    "How do I appeal a grade I think was marked wrong?",
    "Where is the IT helpdesk and what are its hours?",
    "How do I appeal a grade, and where is the IT helpdesk?",
    "لا أستطيع الدخول إلى بوابة الطالب",
    "أريد الانسحاب من مقرر الإحصاء STAT301",
    "How do I withdraw from a course?",
    "I want to withdraw from STAT301, the workload is too heavy",
    "I was told my attendance is short but I have a medical note",
]

print(f"{'DEST':9} {'LANG':5} {'CONF':5}  QUESTION")
print("-" * 100)
for q in ROUTING_TESTS:
    d = router.invoke(q)
    print(f"{d.destination:9} {d.language:5} {d.confidence:5}  {q}")
    print(f"{'':22}reason: {d.reason}")

---
# 10 · Supervisor and workers — **Track A**, rubric §2

This is the declared track's headline evidence: a dedicated supervisor whose only
job is to decide who works next, two specialist workers that do not know about
each other, and **printed `transfer_to_*` tool calls** proving the *LLM* chose
the worker.

Two handoffs per request is correct, not a bug — the supervisor hands off to the
worker, and the worker hands control back with `transfer_back_to_supervisor`.

> `create_agent` takes no `prompt=` argument. Each worker's behaviour comes from
> its **tool docstrings**, which is why those docstrings are written like prompts.

In [ ]:
# create_agent is the current builder; older LangChain exposes create_react_agent.
try:
    from langchain.agents import create_agent
    def make_worker(tools, name):
        return create_agent(model=llm, tools=tools, name=name)
    print("using langchain.agents.create_agent")
except ImportError:
    from langgraph.prebuilt import create_react_agent
    def make_worker(tools, name):
        return create_react_agent(llm, tools=tools, name=name)
    print("using langgraph.prebuilt.create_react_agent (fallback)")

academic_agent = make_worker(
    [search_academic_regulations, compute_gpa, credits_to_graduate],
    "academic_agent")

campus_agent = make_worker(
    [search_campus_services],
    "campus_agent")

print("workers ready:", academic_agent.name, "|", campus_agent.name)

In [ ]:
from langgraph_supervisor import create_supervisor

supervisor = create_supervisor(
    agents=[academic_agent, campus_agent],
    model=llm,
    prompt=(
        "You supervise two Al-Noor University specialists.\n"
        "Route to academic_agent: grades, GPA, exams, attendance, transcripts, "
        "academic probation, course withdrawal rules, graduation requirements.\n"
        "Route to campus_agent: library, IT helpdesk, student portal login, "
        "housing, dining, careers, wellbeing, clubs and societies.\n"
        "After a specialist replies, relay their full answer to the student. "
        "Do not answer from your own knowledge."
    ),
).compile()

print("supervisor compiled")

In [ ]:
# THE DELIVERABLE for rubric section 2: printed handoffs.
SUPERVISOR_TESTS = [
    "What GPA do I need to stay off academic probation?",
    "How late is the library open during finals week?",
]

for q in SUPERVISOR_TESTS:
    result = supervisor.invoke({"messages": [{"role": "user", "content": q}]})
    print(f"\nQ: {q}")
    for m in result["messages"]:
        for tc in getattr(m, "tool_calls", []) or []:
            print("   handoff ->", tc["name"])
    print("   A:", result["messages"][-1].content[:300])

### Why there is also an `@entrypoint` below

The supervisor shape above is a complete, working system — and it is the
declared Track A evidence. It cannot, however, express two things the project
needs:

- an **`action`** branch, where the student is asking to *file* something rather
  than asking a question, and
- a **human approval gate** that pauses before that action is irreversible.

So sections 11–13 wrap the *same tools and the same retrievers* in a LangGraph
Functional API workflow that adds the four-way route, retrieval validation,
retry, memory, and the `interrupt()`. Both layers are real; neither is dead code.

---
# 11 · Short-term and long-term memory — rubric §4

Two different objects, two different lifetimes.

| | Short-term | Long-term |
|---|---|---|
| Object | `InMemorySaver` (checkpointer) | `InMemoryStore` (store) |
| Scoped by | `thread_id` | namespace tuple `("students", student_id)` |
| Survives a new thread? | No | **Yes** |
| Holds | the in-progress run, paused interrupts | language, major, question count |

A growing list of chat messages is **not** long-term memory. The rubric rules
that out explicitly. If it disappears when the thread changes, it was short-term.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

checkpointer = InMemorySaver()    # short-term: the run, keyed by thread_id
store = InMemoryStore()           # long-term: durable facts, keyed by namespace


def remember(student_id: str, key: str, value):
    """Write a durable fact about a student. Independent of any thread."""
    store.put(("students", student_id), key, {"value": value})


def recall(student_id: str, key: str, default=None):
    """Read it back. Returns default if we have never learned it."""
    item = store.get(("students", student_id), key)
    return item.value["value"] if item else default


def load_student_profile(student_id: str) -> dict:
    return {
        "preferred_language": recall(student_id, "preferred_language"),
        "questions_asked": recall(student_id, "questions_asked", 0),
        "last_topic": recall(student_id, "last_topic"),
    }


# quick check of the primitive before wiring it into the workflow
remember("demo", "preferred_language", "ar")
print("recall set   :", recall("demo", "preferred_language"))
print("recall unset :", recall("demo", "major"))
print("checkpointer :", type(checkpointer).__name__)
print("store        :", type(store).__name__)

---
# 12 · The workflow — rubric §6 (Functional API + error handling), §5 (HITL)

Built with `@task` and `@entrypoint`. **No `StateGraph` anywhere** — the rubric
names the Functional API specifically.

Three of the four error strategies are implemented:

| Strategy | Where |
|---|---|
| Transient → `RetryPolicy` | `retrieve_one` |
| LLM-recoverable → loop back with the error | `classify` |
| User-fixable → `interrupt()` | `get_course_code` |
| Unexpected → let it bubble up | no blanket `except` anywhere |

In [ ]:
import inspect
from langgraph.func import entrypoint, task
from langgraph.types import interrupt, Command, RetryPolicy

# The keyword changed between versions: newer is retry_policy=, older is retry=.
# A hand-written for-loop with time.sleep() is NOT a RetryPolicy and earns nothing.
RETRY_KW = ("retry_policy" if "retry_policy" in inspect.signature(task).parameters
            else "retry")
RETRY = {RETRY_KW: RetryPolicy(max_attempts=3, initial_interval=0.5)}
print("retry keyword for this langgraph version:", RETRY_KW)

In [ ]:
from pydantic import ValidationError

# ---- guardrail at the boundary ------------------------------------------
class Query(BaseModel):
    """Rejects malformed input before it reaches an LLM call."""
    student_id: str = Field(min_length=3, max_length=20)
    question: str = Field(min_length=1, max_length=2000)


# ---- error strategy 2: LLM-recoverable ----------------------------------
@task
def classify(question: str) -> MurshidRoute:
    """Route the question. On invalid model output, re-prompt WITH the error."""
    feedback = ""
    for attempt in range(3):
        try:
            return router.invoke(
                f"Route this student question.{feedback}\n\n{question}")
        except ValidationError as e:
            print(f"  [classify] invalid output on attempt {attempt + 1}, re-prompting")
            feedback = (f"\n\nYour previous answer was rejected: {e}. "
                        f"destination must be exactly one of: "
                        f"academic, campus, both, action.")
    # Exhausted. Return a SAFE default rather than crashing — and note that
    # confidence='low' routes this straight to a human instead of guessing.
    print("  [classify] exhausted retries -> escalating to a human")
    return MurshidRoute(destination="both", language="en", confidence="low",
                        reason="classifier could not produce valid output")


# ---- error strategy 1: transient -----------------------------------------
SIMULATE_FLAKY = False          # flipped to True in section 17 to prove the retry
RETRIEVAL_ATTEMPTS = {"count": 0}


@task(**RETRY)
def retrieve_one(question: str, source: str) -> str:
    """Retrieve from ONE store. Carries a real RetryPolicy."""
    RETRIEVAL_ATTEMPTS["count"] += 1
    if SIMULATE_FLAKY and RETRIEVAL_ATTEMPTS["count"] == 1:
        print(f"  [retrieve_one] attempt #{RETRIEVAL_ATTEMPTS['count']} — raising")
        raise ConnectionError("simulated transient vector-store timeout")
    print(f"  [retrieve_one] attempt #{RETRIEVAL_ATTEMPTS['count']} ({source})")

    retriever = academic_retriever if source == "academic" else campus_retriever
    docs = retriever.invoke(question)
    if not docs:
        return ""
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)


@task
def synthesize(question: str, context: str, profile: dict,
               route: MurshidRoute) -> str:
    """Write the student-facing answer from the retrieved context only."""
    lang = profile.get("preferred_language") or route.language
    lang_name = "Arabic" if lang == "ar" else "English"
    if not context.strip():
        return ("I could not find that in the Al-Noor University documents I "
                "have access to. Please contact Student Services directly.")
    return llm.invoke(
        f"You are Murshid, an Al-Noor University student services assistant.\n"
        f"Answer the student's question using ONLY the context below. "
        f"If the context does not contain the answer, say so plainly.\n"
        f"Cite the source file in square brackets. Reply in {lang_name}.\n\n"
        f"CONTEXT:\n{context}\n\nQUESTION: {question}"
    ).content

In [ ]:
# ---- the irreversible actions -------------------------------------------
REGISTRY_LOG = []      # stands in for the Registrar system


@task
def submit_withdrawal(student_id: str, course_code: str, reason: str) -> str:
    """File a course withdrawal. IRREVERSIBLE once the Registrar processes it."""
    ref = f"WD-{student_id}-{course_code}"
    REGISTRY_LOG.append({"ref": ref, "type": "withdrawal", "student": student_id,
                         "course": course_code, "reason": reason})
    return (f"Withdrawal {ref} filed for {course_code}. "
            f"Recorded reason: {reason}")


# ---- error strategy 3: user-fixable -------------------------------------
@task
def get_course_code(code: str | None) -> str:
    """Pause and ask if the student did not name a course."""
    if not code:
        code = interrupt({"message": "Which course code do you want to withdraw from?",
                          "field": "course_code"})
    return str(code).upper().strip()


class ActionRequest(BaseModel):
    """What the student is asking us to file."""
    action_type: Literal["withdrawal", "appeal"]
    course_code: str | None = Field(
        default=None, description="e.g. STAT301; null if the student did not say")
    reason: str = Field(description="The student's stated reason, in their words")


@task
def parse_action(question: str) -> ActionRequest:
    return llm.with_structured_output(ActionRequest).invoke(
        f"Extract the formal request from this student message:\n\n{question}")


# ---- rubric section 5: the human approval gate --------------------------
@task
def request_approval(payload: dict) -> dict:
    """Pause the run until a student advisor approves, edits, or rejects."""
    return interrupt({
        "action": "A student advisor must approve this before it is filed",
        **payload,
    })

In [ ]:
@entrypoint(checkpointer=checkpointer, store=store)
def murshid(inputs: dict) -> dict:
    q = Query(**inputs)                                   # guardrail
    profile = load_student_profile(q.student_id)          # LONG-TERM READ

    route = classify(q.question).result()                 # §2 LLM routing

    # ---------------- action branch: pause for a human ------------------
    if route.destination == "action" or route.confidence == "low":
        req = parse_action(q.question).result()
        course = get_course_code(req.course_code).result()   # may interrupt

        decision = request_approval({
            "type": req.action_type,
            "student_id": q.student_id,
            "course_code": course,
            "student_stated_reason": req.reason,
            "router_confidence": route.confidence,
        }).result()

        if not decision.get("approved"):
            answer = ("Your request was not filed. Advisor note: "
                      + decision.get("note", "no reason given"))
            outcome = "rejected_by_advisor"
        else:
            # The ADVISOR's edit is what reaches the registry, not the student's text.
            answer = submit_withdrawal(
                q.student_id,
                decision.get("course_code", course),
                decision.get("edited_reason", req.reason),
            ).result()
            outcome = "filed"
        searched = []

    # ---------------- both: Parallelization ------------------------------
    elif route.destination == "both":
        a_fut = retrieve_one(q.question, "academic")      # launched...
        c_fut = retrieve_one(q.question, "campus")        # ...concurrently
        context = a_fut.result() + "\n\n" + c_fut.result()
        searched = ["academic", "campus"]
        answer = synthesize(q.question, context, profile, route).result()
        outcome = "answered"

    # ---------------- single source, with retrieval validation ----------
    else:
        context = retrieve_one(q.question, route.destination).result()
        searched = [route.destination]
        if not context.strip():                           # Hybrid RAG validation
            other = "campus" if route.destination == "academic" else "academic"
            print(f"  [validate] {route.destination} returned nothing, trying {other}")
            context = retrieve_one(q.question, other).result()
            searched.append(other)
        answer = synthesize(q.question, context, profile, route).result()
        outcome = "answered"

    # ---------------- LONG-TERM WRITE -----------------------------------
    asked = profile["questions_asked"] + 1
    remember(q.student_id, "questions_asked", asked)
    remember(q.student_id, "preferred_language", route.language)
    remember(q.student_id, "last_topic", route.destination)

    return {
        "answer": answer,
        "outcome": outcome,
        "routed_to": route.destination,
        "reason": route.reason,
        "sources_searched": searched,
        "questions_asked_total": asked,
    }


print("workflow compiled — tasks:",
      "classify, retrieve_one, synthesize, parse_action, get_course_code, "
      "request_approval, submit_withdrawal")

---
# 13 · Demo A — an academic question

In [ ]:
r = murshid.invoke(
    {"student_id": "s2201", "question": "How long do I have to appeal a grade?"},
    {"configurable": {"thread_id": "demo-academic"}})

print("routed to      :", r["routed_to"])
print("reason         :", r["reason"])
print("sources searched:", r["sources_searched"])
print("\n" + r["answer"])

---
# 14 · Demo B — a campus question, asked in Arabic

No English keyword appears anywhere in this question. A keyword router returns
nothing; the LLM classifier routes it correctly and the answer comes back in
Arabic.

In [ ]:
r = murshid.invoke(
    {"student_id": "s2301", "question": "لا أستطيع الدخول إلى بوابة الطالب، ماذا أفعل؟"},
    {"configurable": {"thread_id": "demo-campus-ar"}})

print("routed to      :", r["routed_to"])
print("reason         :", r["reason"])
print("sources searched:", r["sources_searched"])
print("\n" + r["answer"])

---
# 15 · Demo C — a question spanning both sources

`sources_searched` should be `['academic', 'campus']`. The two retrievals are
independent, so they are launched as concurrent tasks and awaited together —
the **Parallelization** pattern inside the `both` branch.

In [ ]:
r = murshid.invoke(
    {"student_id": "s2201",
     "question": "How do I appeal a grade, and where is the IT helpdesk?"},
    {"configurable": {"thread_id": "demo-both"}})

print("routed to      :", r["routed_to"])
print("sources searched:", r["sources_searched"])
print("\n" + r["answer"])

---
# 16 · Demo D — human-in-the-loop: interrupt **and** resume — rubric §5

A course withdrawal cannot be reinstated in the same term once the Registrar
processes it, and the deadline is the end of week 10. That is a real reason to
require a human, not a decorative one.

**Three things must be visible in the output below**, and all three are checked:

1. the run paused (`__interrupt__` printed),
2. `Command(resume=...)` completed it,
3. the **advisor's edited text** — not the student's original — reached the
   registry.

⚠️ The same `thread_id` is used for both calls. That is how the checkpointer
finds the paused run.

In [ ]:
cfg = {"configurable": {"thread_id": "withdrawal-1"}}

paused = murshid.invoke(
    {"student_id": "s2201",
     "question": "I want to withdraw from STAT301, the workload is too heavy"},
    cfg)

print("=== PAUSED FOR ADVISOR APPROVAL ===")
for k, v in paused["__interrupt__"][0].value.items():
    print(f"  {k:22}: {v}")

In [ ]:
# The advisor approves, but REWRITES the reason before it is filed.
done = murshid.invoke(Command(resume={
    "approved": True,
    "course_code": "STAT301",
    "edited_reason": "Medical grounds — documentation on file with Student Health.",
}), cfg)

print("=== RESUMED AND COMPLETED ===")
print("outcome:", done["outcome"])
print("answer :", done["answer"])

print("\n=== REGISTRY LOG ===")
for entry in REGISTRY_LOG:
    print(" ", entry)

print("\nThe advisor's text reached the registry, not the student's original "
      "'the workload is too heavy'. That is the difference between a real "
      "handoff and a decorative pause.")

---
# 17 · Demo E — the rejection path

Both branches of the approval gate, not just the happy one.

In [ ]:
cfg_reject = {"configurable": {"thread_id": "withdrawal-2"}}

paused = murshid.invoke(
    {"student_id": "s2450",
     "question": "Please withdraw me from CHEM210, I've stopped attending"},
    cfg_reject)
print("PAUSED:", paused["__interrupt__"][0].value["course_code"])

rejected = murshid.invoke(Command(resume={
    "approved": False,
    "note": "Past the end-of-week-10 deadline; withdrawal cannot be processed.",
}), cfg_reject)

print("\noutcome:", rejected["outcome"])
print("answer :", rejected["answer"])
print("\nregistry entries:", len(REGISTRY_LOG), "— unchanged, nothing was filed.")

---
# 18 · Cross-thread memory proof — rubric §4

The **only** thing that distinguishes a Store from a chat history: write a fact
in one thread, read it back from a **completely different** thread.

If `conv-B` prints `1`, the value was living in the thread rather than the store,
and it was never long-term memory.

In [ ]:
SID = "s9001"

# --- thread A: this student asks in Arabic for the first time ---
r1 = murshid.invoke({"student_id": SID, "question": "ما هو الحد الأدنى لنسبة الحضور؟"},
                    {"configurable": {"thread_id": "conv-A"}})
print("conv-A  questions_asked:", r1["questions_asked_total"],
      "| language learned:", recall(SID, "preferred_language"))

# --- thread B: A COMPLETELY DIFFERENT THREAD, same student ---
r2 = murshid.invoke({"student_id": SID, "question": "And the withdrawal deadline?"},
                    {"configurable": {"thread_id": "conv-B"}})
print("conv-B  questions_asked:", r2["questions_asked_total"],
      "| language recalled:", recall(SID, "preferred_language"),
      "  <- survived a brand-new thread")

# --- thread C: a different student starts from zero ---
r3 = murshid.invoke({"student_id": "s9002", "question": "Where is the careers office?"},
                    {"configurable": {"thread_id": "conv-C"}})
print("conv-C  questions_asked:", r3["questions_asked_total"], "(different student)")

print("\nExpected: conv-A -> 1, conv-B -> 2, conv-C -> 1")
print("\nNote conv-B's answer came back in Arabic without the student asking,")
print("because preferred_language was recalled from the Store:")
print(r2["answer"][:300])

### Short-term memory — the other half of §4

Two turns in the **same** `thread_id`. The checkpointer is what keeps the run's
state between them.

In [ ]:
cfg_short = {"configurable": {"thread_id": "conv-short-term"}}

t1 = murshid.invoke({"student_id": "s9003",
                     "question": "What is the minimum attendance percentage?"}, cfg_short)
print("turn 1:", t1["answer"][:200], "\n")

t2 = murshid.invoke({"student_id": "s9003",
                     "question": "What happens if I fall below it?"}, cfg_short)
print("turn 2:", t2["answer"][:300])
print("\nquestions_asked_total across the two turns:", t2["questions_asked_total"])

---
# 19 · Proving the retry actually fires — rubric §6

A `RetryPolicy` that exists but never runs proves nothing. Below, the first
attempt raises `ConnectionError` on purpose. The expected output is **attempt #1
followed by attempt #2**, with no error surfacing and no retry code of our own.

In [ ]:
SIMULATE_FLAKY = True
RETRIEVAL_ATTEMPTS["count"] = 0

r = murshid.invoke(
    {"student_id": "s9100", "question": "How many credit hours do I need to graduate?"},
    {"configurable": {"thread_id": "retry-demo"}})

SIMULATE_FLAKY = False

print("\ntotal retrieval attempts:", RETRIEVAL_ATTEMPTS["count"])
print("answer still returned successfully:", bool(r["answer"]))
print("\n" + r["answer"][:250])

---
# 20 · LangSmith — flush and evaluate — rubric §8

Traces are sent in the background, so flush before you go looking.

In [ ]:
if TRACING_ON:
    from langchain_core.tracers.langchain import wait_for_all_tracers
    wait_for_all_tracers()
    print("Flushed. Open https://smith.langchain.com and select project:",
          os.environ["LANGCHAIN_PROJECT"])
    print("\nIn the trace, find these four things — they are what your write-up "
          "should describe:")
    print("  1. the tree shape — does it match what you intended?")
    print("  2. latency per task — which one dominates?")
    print("  3. token counts and cost, per call and totalled")
    print("  4. the exact input and output of each step")
else:
    print("Tracing is off — no key. Section 8 of the write-up must say so plainly")
    print("and describe what you inspected instead (printed handoffs, the")
    print("retrieval attempt counts, sources_searched). Do NOT claim a trace")
    print("finding you did not observe.")

In [ ]:
# Evaluation: score the router against a small dataset.
if TRACING_ON:
    DATASET = "murshid-routing-tests"
    examples = [
        ("How do I appeal a grade?",                        "academic"),
        ("What are the library hours during finals?",       "campus"),
        ("لا أستطيع الدخول إلى بوابة الطالب",                 "campus"),
        ("ما هي شروط التخرج؟",                                "academic"),
        ("I want to withdraw from STAT301 right now",       "action"),
        ("How do I appeal a grade and where is IT support?", "both"),
    ]

    if not client.has_dataset(dataset_name=DATASET):
        ds = client.create_dataset(dataset_name=DATASET)
        client.create_examples(
            inputs=[{"question": q} for q, _ in examples],
            outputs=[{"expected_destination": d} for _, d in examples],
            dataset_id=ds.id)
        print("created dataset:", DATASET)
    else:
        print("dataset already exists:", DATASET)

    def route_only(inputs: dict) -> dict:
        return {"destination": router.invoke(inputs["question"]).destination}

    def routes_correctly(outputs: dict, reference_outputs: dict) -> bool:
        """Deterministic grader — cheap, and no LLM judge to second-guess."""
        return outputs["destination"] == reference_outputs["expected_destination"]

    results = client.evaluate(route_only, data=DATASET,
                              evaluators=[routes_correctly],
                              experiment_prefix="murshid-routing")
    print("Open the experiment in LangSmith to compare runs side by side.")
else:
    print("Skipped — no LangSmith key.")

---
# 21 · Write-up

The full write-up lives in [`WRITEUP.md`](../WRITEUP.md) — one paragraph per
rubric section.

**Write it from the output above, not from your plan.** Graders check it against
the code and the captured output, and a claim your own notebook contradicts
costs more than an admitted gap.

## Before you submit

- [ ] Your **full name** is in the header of this notebook and in the README
- [ ] **Restart the kernel and run everything top to bottom**, in order
- [ ] Every demo cell has **captured output saved in the file**
- [ ] The interrupt **and** the resume both ran (§16), and the advisor's edit is
      visible in the registry log
- [ ] The cross-thread test printed conv-A → 1, conv-B → 2, conv-C → 1 (§18)
- [ ] The retry printed attempt #1 → attempt #2 (§19)
- [ ] `grep -rn "«" .` returns nothing — no placeholder text left anywhere
- [ ] No API key in any cell, and none in git history
- [ ] `.gitignore` excludes `.env`, `*.db`, `chroma/`
- [ ] Programme name and cohort dates stated in the README
- [ ] **Declared track (A) stated explicitly** — it is, at the top of this notebook
- [ ] Every claim in `WRITEUP.md` is backed by something visible above